In [ ]:
# Check dataset structure
import pandas as pd
import numpy as np
df= pd.read_csv("retail_sales.csv")
df.shape
df.head()
df.info()

In [4]:
# checking missing value brfore cleaning
df.isnull().sum()

Transaction ID         0
Customer ID            0
Category               0
Item                1213
Price Per Unit       609
Quantity             604
Total Spent          604
Payment Method         0
Location               0
Transaction Date       0
Discount Applied    4199
dtype: int64

In [5]:
# Handling missing values : Item
df["Item"] = df["Item"].fillna("Unknown")

In [6]:
# Handling Price per Unit
df["Price Per Unit"] = df["Price Per Unit"].fillna(
    df["Price Per Unit"].median()
)

In [7]:
# Handling quantity
df["Quantity"] = df["Quantity"].fillna(
    df["Quantity"].median()
)

In [8]:
# Handling total spent
df["Total Spent"] = df["Total Spent"].fillna(
    df["Price Per Unit"] * df["Quantity"]
)

In [9]:
# Handling discount applied
df["Discount Applied"] = df["Discount Applied"].fillna("Unknown")

In [10]:
df.isnull().sum()

Transaction ID      0
Customer ID         0
Category            0
Item                0
Price Per Unit      0
Quantity            0
Total Spent         0
Payment Method      0
Location            0
Transaction Date    0
Discount Applied    0
dtype: int64

In [11]:
# Checking for duplicates
df.duplicated().sum()

np.int64(0)

In [12]:
duplicate_count = df.duplicated().sum()
print("Number of duplicate records:", duplicate_count)

Number of duplicate records: 0


In [14]:
print(df.dtypes)

Transaction ID          str
Customer ID             str
Category                str
Item                    str
Price Per Unit      float64
Quantity            float64
Total Spent         float64
Payment Method          str
Location                str
Transaction Date        str
Discount Applied     object
dtype: object


In [15]:
# converting datatype
df["Transaction Date"] = pd.to_datetime(
    df["Transaction Date"],
    errors="coerce"
)

In [16]:
df["Transaction Date"].dtype

dtype('<M8[us]')

In [17]:
df["Transaction Date"].isnull().sum()

np.int64(0)

In [18]:
# Extract year
df["Year"] = df["Transaction Date"].dt.year

In [19]:
# Extarct month
df["Month"] = df["Transaction Date"].dt.month

In [20]:
# Extract month name
df["Month Name"] = df["Transaction Date"].dt.month_name()

In [21]:
df[["Transaction Date", "Year", "Month", "Month Name"]].head(10)

,Transaction Date,Year,Month,Month Name
0,2024-04-08,2024,4,April
1,2023-07-23,2023,7,July
2,2022-10-05,2022,10,October
3,2022-05-07,2022,5,May
4,2022-10-02,2022,10,October
5,2023-11-30,2023,11,November
6,2023-06-10,2023,6,June
7,2024-04-02,2024,4,April
8,2023-04-26,2023,4,April
9,2024-03-14,2024,3,March


In [22]:
# Checking outliers
df[["Price Per Unit", "Quantity", "Total Spent"]].describe()

,Price Per Unit,Quantity,Total Spent
count,12575.000000,12575.000000,12575.000000
mean,23.348191,5.558648,130.208111
std,10.480413,2.790160,93.580667
min,5.000000,1.000000,5.000000
25%,14.000000,3.000000,52.000000
50%,23.000000,6.000000,110.000000
75%,32.000000,8.000000,192.000000
max,41.000000,10.000000,410.000000


In [23]:
# Calculating IQR for all 3 columns
Q1 = df[["Price Per Unit", "Quantity", "Total Spent"]].quantile(0.25)
Q3 = df[["Price Per Unit", "Quantity", "Total Spent"]].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("Lower Bounds:")
print(lower_bound)

print("\nUpper Bounds:")
print(upper_bound)

Lower Bounds:
Price Per Unit    -13.0
Quantity           -4.5
Total Spent      -158.0
dtype: float64

Upper Bounds:
Price Per Unit     59.0
Quantity           15.5
Total Spent       402.0
dtype: float64


In [24]:
# Count outliers
outliers_total_spent = df[
    (df["Total Spent"] < lower_bound["Total Spent"]) |
    (df["Total Spent"] > upper_bound["Total Spent"])
]

print("Total Spent outliers:", len(outliers_total_spent))

Total Spent outliers: 60


In [25]:
outliers_total_spent[["Price Per Unit", "Quantity", "Total Spent"]]

,Price Per Unit,Quantity,Total Spent
27,41.0,10.0,410.0
133,41.0,10.0,410.0
339,41.0,10.0,410.0
869,41.0,10.0,410.0
1060,41.0,10.0,410.0
1088,41.0,10.0,410.0
1468,41.0,10.0,410.0
1505,41.0,10.0,410.0
1568,41.0,10.0,410.0
1950,41.0,10.0,410.0


In [26]:
outliers_total_spent["Total Spent"].describe()

count     60.0
mean     410.0
std        0.0
min      410.0
25%      410.0
50%      410.0
75%      410.0
max      410.0
Name: Total Spent, dtype: float64

In [27]:
# Inconsistent categorical data
df["Category"].value_counts()

Category
Furniture                             1591
Electric household essentials         1591
Food                                  1588
Milk Products                         1584
Butchers                              1568
Beverages                             1567
Computers and electric accessories    1558
Patisserie                            1528
Name: count, dtype: int64

In [28]:
df["Payment Method"].value_counts()

Payment Method
Cash              4310
Digital Wallet    4144
Credit Card       4121
Name: count, dtype: int64

In [29]:
df["Location"].value_counts()

Location
Online      6354
In-store    6221
Name: count, dtype: int64

In [30]:
df["Discount Applied"].value_counts(dropna=False)

Discount Applied
True       4219
Unknown    4199
False      4157
Name: count, dtype: int64

In [32]:
df["Discount Applied"] = df["Discount Applied"].astype("category")

In [33]:
df["Discount Applied"].dtype

CategoricalDtype(categories=[False, True, 'Unknown'], ordered=False, categories_dtype=object)

In [34]:
# Final data validation
df.info()
df.isnull().sum()
df.duplicated().sum()
df.shape

<class 'pandas.DataFrame'>
RangeIndex: 12575 entries, 0 to 12574
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Transaction ID    12575 non-null  str           
 1   Customer ID       12575 non-null  str           
 2   Category          12575 non-null  str           
 3   Item              12575 non-null  str           
 4   Price Per Unit    12575 non-null  float64       
 5   Quantity          12575 non-null  float64       
 6   Total Spent       12575 non-null  float64       
 7   Payment Method    12575 non-null  str           
 8   Location          12575 non-null  str           
 9   Transaction Date  12575 non-null  datetime64[us]
 10  Discount Applied  12575 non-null  category      
 11  Year              12575 non-null  int32         
 12  Month             12575 non-null  int32         
 13  Month Name        12575 non-null  str           
dtypes: category(1), datetime64[us](1)

(12575, 14)

In [35]:
print("Missing values:")
print(df.isnull().sum())

print("\nDuplicate records:")
print(df.duplicated().sum())

print("\nDataset shape:")
print(df.shape)

Missing values:
Transaction ID      0
Customer ID         0
Category            0
Item                0
Price Per Unit      0
Quantity            0
Total Spent         0
Payment Method      0
Location            0
Transaction Date    0
Discount Applied    0
Year                0
Month               0
Month Name          0
dtype: int64

Duplicate records:
0

Dataset shape:
(12575, 14)


In [36]:
df.to_csv("clean_dataset.csv", index=False)
print("Clean dataset exported successfully!")

Clean dataset exported successfully!


In [37]:
# Export Clean Dataset
import os

print(os.path.exists("clean_dataset.csv"))

True


In [38]:
clean_df = pd.read_csv("clean_dataset.csv")

print("Shape:", clean_df.shape)
print("\nMissing values:")
print(clean_df.isnull().sum())

Shape: (12575, 14)

Missing values:
Transaction ID      0
Customer ID         0
Category            0
Item                0
Price Per Unit      0
Quantity            0
Total Spent         0
Payment Method      0
Location            0
Transaction Date    0
Discount Applied    0
Year                0
Month               0
Month Name          0
dtype: int64


In [ ]:
# ============================================================
# PROFIT MARGIN FEATURE
# ============================================================
# The dataset does not contain a Cost or Profit column.
# Therefore, Profit Margin cannot be calculated accurately
# without making an unsupported assumption about product cost.
# Profit Margin = (Profit / Revenue) * 100
# Profit = Revenue - Cost
# Since Cost is unavailable, Profit Margin was not calculated.